# Sprint 0 — 資料驗證

使用 DuckDB 對原始 CSV 做基本驗證，確認欄位格式、筆數、JOIN 可行性。

> ⚠️ **本 Notebook 純屬驗證用途，不產生任何輸出檔案，不需重跑。**

In [1]:
import duckdb

con = duckdb.connect()  # in-memory connection，不載入整表

# 資料路徑
PATHS = {
    'Order_TG'  : '91APP_Dataset(main)/Order_TG.csv',
    'Member'    : '91APP_Dataset(main)/Member.csv',
    'session01' : '91APP_Dataset(session01)/session01_202309.csv',
    'session02' : '91APP_Dataset(session02)/session02_202309.csv',
}

## 1. DESCRIBE — 欄位名稱與型別

列出 Order_TG、Member、session01、session02 的所有欄位與推斷型別，確認格式符合預期。

> ℹ️ 不產生輸出檔案

In [2]:
for name, path in PATHS.items():
    print(f"\n{'='*60}")
    print(f"  {name}  →  {path}")
    print('='*60)
    df = con.execute(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").df()
    print(df[['column_name', 'column_type']].to_string(index=False))


  Order_TG  →  91APP_Dataset(main)/Order_TG.csv
              column_name column_type
                   ShopId     VARCHAR
             ShopMemberId     VARCHAR
          TradesGroupCode     VARCHAR
            OrderDateTime   TIMESTAMP
              ChannelType     VARCHAR
            ChannelDetail     VARCHAR
              PaymentType     VARCHAR
             ShippingType     VARCHAR
                  TsCount      BIGINT
                      Qty      BIGINT
         TotalSalesAmount      DOUBLE
               TotalPrice      DOUBLE
            TotalDiscount      DOUBLE
   TotalPromotionDiscount      DOUBLE
      TotalCouponDiscount      DOUBLE
TotalLoyaltyPointDiscount      DOUBLE
                StatusDef     VARCHAR

  Member  →  91APP_Dataset(main)/Member.csv
             column_name column_type
                  ShopId     VARCHAR
            ShopMemberId     VARCHAR
   RegisterSourceTypeDef     VARCHAR
        RegisterDateTime   TIMESTAMP
                  Gender     VARCHAR


## 2. Row Count — 各表筆數

確認四張資料表的原始筆數，作為後續篩選比例的基準值。

> ℹ️ 不產生輸出檔案

In [3]:
print(f"{'Table':<12} {'Row Count':>12}")
print('-' * 26)
for name, path in PATHS.items():
    cnt = con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{path}')").fetchone()[0]
    print(f"{name:<12} {cnt:>12,}")

Table           Row Count
--------------------------


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Order_TG       11,470,535
Member          6,907,009


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

session01      13,796,942
session02       1,672,607


## 3. ShopMemberId 型別一致性確認（Order_TG vs Member）

確認兩表的 `ShopMemberId` 欄位型別相同（皆為 VARCHAR），可直接 JOIN 而不需型別轉換。

> ℹ️ 不產生輸出檔案

In [4]:
def get_col_type(path, col):
    rows = con.execute(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").fetchall()
    for r in rows:
        if r[0] == col:
            return r[1]
    return None

type_order  = get_col_type(PATHS['Order_TG'], 'ShopMemberId')
type_member = get_col_type(PATHS['Member'],   'ShopMemberId')

print(f"Order_TG .ShopMemberId type : {type_order}")
print(f"Member   .ShopMemberId type : {type_member}")
print()
if type_order == type_member:
    print("[OK] 型別一致，可直接 JOIN。")
else:
    print("[WARNING] 型別不一致，JOIN 前需做型別轉換！")

Order_TG .ShopMemberId type : VARCHAR
Member   .ShopMemberId type : VARCHAR

[OK] 型別一致，可直接 JOIN。


## 4. Member JOIN Order_TG — 取前 5 筆確認能對上

執行 Member × Order_TG 的 INNER JOIN，取前 5 筆確認 ShopMemberId 能正確對應，JOIN 結果符合預期。

> ℹ️ 不產生輸出檔案

In [5]:
sql = f"""
SELECT
    m.ShopMemberId,
    m.Gender,
    m.MemberCardLevel,
    o.OrderDateTime,
    o.ChannelType,
    o.TotalSalesAmount
FROM
    read_csv_auto('{PATHS["Member"]}')   AS m
    INNER JOIN read_csv_auto('{PATHS["Order_TG"]}') AS o
        ON m.ShopMemberId = o.ShopMemberId
LIMIT 5
"""

result = con.execute(sql).df()
print(f"JOIN 成功，回傳 {len(result)} 筆")
result

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

JOIN 成功，回傳 5 筆


,ShopMemberId,Gender,MemberCardLevel,OrderDateTime,ChannelType,TotalSalesAmount
0,3BSOU+Cl0KLgu0XBtlZ4F4ZViyYEgTgrfrCne/B7nz0=,Female,20,2023-07-09 14:00:34,Pos,187.0
1,WhxHcm1gS+pevRBM0HSMexGpezd48ITPTDoqcAl1nIk=,Female,10,2023-02-23 15:36:49,Pos,99.0
2,tt1FEGKHEKDVNzgyX9y5D/L28idjq4FwRmTcz3v1N4Y=,Male,30,2023-06-21 17:40:07,Pos,952.0
3,deTCAa8FVjR8CM72ruTbcc7B5KKhIj3veUX2wld7aJw=,Male,20,2023-10-10 15:44:50,Pos,2520.0
4,WCrGXHQHGIWVC+ZgY8LDd5+Q8zLl97xt5BQlD6/o8QI=,Female,10,2024-02-13 11:07:08,Pos,541.0
